# MNet + MSU + CBAM for DRIVE dataset (w augs, newDataloader)

In [12]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys
sys.path.append('/content/drive/MyDrive/UNet Testing Template src/')
import src.training.metrics as m
print(m)

Mounted at /content/drive
<module 'src.training.metrics' from '/content/drive/MyDrive/UNet Testing Template src/src/training/metrics.py'>


In [13]:
from pathlib import Path
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# CHANGE THESE

In [ ]:
TITLE = "MNet + MSU + CBAM for DRIVE dataset (w augs, newDataloader)"

SAVE_FILE = "mnet_msu_cbam_w_aug_newDataloader_drive.pth"

SAVE_DIR = Path("/content/drive/MyDrive/UNet Testing Template src/pth")

DATA_DIR = "/content/drive/MyDrive/Retinal_Vessel_Segmentation_Datasets/DRIVE/"

LABEL_FOLDER = "1st_manual"


from models.unet_exp.mnet_exp.mnet_msu_cbam import UNetWithMSUSkipsCBAM_MNet

#pos_prior = p if 'p' in locals() else 0.10      # If you estimated 'p' earlier via estimate_class_weights, use it; else default to 0.10

model_core = UNetWithMSUSkipsCBAM_MNet(
    in_channels=1,
    cbam_reduction=16,
    cbam_use_spatial=True,
    cbam_on_bottleneck=True,
    use_deep_supervision=False,       # start False
    return_side_outputs=False
).to(DEVICE)

# IMPORTS

In [15]:
import os, json, time, math, random
from pathlib import Path

import numpy as np
import torch.nn as nn
import torch.nn.functional as F

# --- data utils ---
from src.data.prepare_dataset import (
    build_pairs_for_split,
    build_all_train_pairs,
    assert_dataset_layout,
    sanity_check_sample_alignment,
)
from src.data.dataloader import make_loaders
from src.data.augmentations import get_train_augs, get_val_augs


# --- evaluation & visualization ---
from src.evaluation.evaluate import evaluate_and_print
from src.evaluation.visualization import visualize_samples

# SET SEED

In [16]:
SEED = 1337
SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
torch.backends.cudnn.benchmark = True

# MODEL PARAMS

In [17]:

# param count
print(sum(x.numel() for x in model_core.parameters() if x.requires_grad)/1e6, "M params")

31.261545 M params


# DATALOADERS

In [18]:
# 1) Pairs
train_pairs_full = build_pairs_for_split(DATA_DIR, split="training", label_folder=LABEL_FOLDER)   # 20 imgs
test_pairs       = build_pairs_for_split(DATA_DIR, split="test",     label_folder=LABEL_FOLDER)   # 20 imgs

# 2) Train/Val loaders: let make_loaders do an 80/20 split from training
#    (this is your "validation" during training; NOT the test set)
train_loader, val_loader = make_loaders(
    train_pairs=train_pairs_full,
    val_pairs=None,               # <-- auto 80/20 split from training
    image_size=512,
    batch_size=2,
    num_workers=1,                # Windows/Jupyter: keep 0
    seed=1337,
    strict_fov=True,
    augs_train=get_train_augs(512),
    augs_val=get_val_augs(512),   # deterministic, no random augs
)

# 3) Test loader: separate; same preprocessing as val (no random augs)
_, test_loader = make_loaders(
    train_pairs=test_pairs,       # dummy (ignored) – we just want the second loader built
    val_pairs=test_pairs,         # pass test_pairs so the returned "val" loader = test loader
    image_size=512,
    batch_size=2,
    num_workers=1,
    seed=1337,
    strict_fov=True,
    augs_train=None,              # unused
    augs_val=get_val_augs(512),   # no random augs
)

# LOSS

In [19]:
from src.training.loss_functions import DiceBCEComplementLoss
import torch

# (optional) estimate class weights from a few batches to handle heavy imbalance
def estimate_class_weights(loader, max_batches=20):
    with torch.no_grad():
        s = 0.0; n = 0
        for i, b in enumerate(loader):
            s += b["mask"].float().mean().item()  # foreground fraction in [0,1]
            n += 1
            if i+1 >= max_batches: break
    p = max(1e-6, min(1-1e-6, s / max(1, n)))      # foreground prior
    w1 = 1.0 / p                                    # weight FG inversely to its freq
    w0 = 1.0 / (1.0 - p)                            # weight BG inversely to its freq
    # normalize so w0 + w1 ≈ 2 (keeps scale stable)
    s2 = w0 + w1
    w0 = 2.0 * w0 / s2; w1 = 2.0 * w1 / s2
    return w0, w1, p

w0, w1, p = estimate_class_weights(train_loader, max_batches=20)
print(f"~ foreground prior p ≈ {p:.5f} | class weights -> w0={w0:.3f}, w1={w1:.3f}")

loss_fn = DiceBCEComplementLoss(
    w0=w0, w1=w1,
    dice_weight=0.5,         # tune (e.g., 0.6 Dice / 0.4 BCE)
    bce_weight=0.5,
    exact_equation=False,    # keep False so the BCE term is minimized (not added)
    reduction="mean",
)

~ foreground prior p ≈ 0.08617 | class weights -> w0=0.172, w1=1.828


# TRAIN LOOP

In [20]:
def train_one_epoch(model, loader, optimizer, scaler, loss_fn):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        x = batch["image"].to(DEVICE, non_blocking=True)  # [B,1,H,W]
        y = batch["mask"].to(DEVICE,  non_blocking=True)  # [B,1,H,W]

        # Optional: mask labels outside FOV if present (keeps loss fair)
        if "fov" in batch:
            fov = batch["fov"].to(DEVICE, non_blocking=True)
            y = y * (fov > 0.5).float()

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE=="cuda")):
            logits = model(x)          # ← PURE UNET: no fov passed
            loss   = loss_fn(logits, y)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total += loss.item() * x.size(0)
        n     += x.size(0)
    return total / max(1, n)

@torch.no_grad()
def validate_loss_only(model, loader, loss_fn):
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        x = batch["image"].to(DEVICE, non_blocking=True)
        y = batch["mask"].to(DEVICE,  non_blocking=True)

        if "fov" in batch:
            fov = batch["fov"].to(DEVICE, non_blocking=True)
            y = y * (fov > 0.5).float()

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE=="cuda")):
            logits = model(x)
            loss   = loss_fn(logits, y)

        total += loss.item() * x.size(0)
        n     += x.size(0)
    return total / max(1, n)

In [ ]:
import json, time, math
from torch.optim.lr_scheduler import LambdaLR

# --- optimizer / scaler / scheduler ---
optimizer = torch.optim.AdamW(model_core.parameters(), lr=3e-4, weight_decay=1e-4)
scaler    = torch.amp.GradScaler("cuda", enabled=(DEVICE=="cuda"))

EPOCHS = 100
warmup_epochs = 5

def lr_lambda(e):
    if e < warmup_epochs:
        return (e + 1) / warmup_epochs
    progress = (e - warmup_epochs) / max(1, EPOCHS - warmup_epochs)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda)

best_val = float("inf")

# --- lists to collect losses for plotting later ---
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS+1):
    print(f"----- EPOCH {epoch} -----")
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.time()

    tr_loss  = train_one_epoch(model_core, train_loader, optimizer, scaler, loss_fn)
    val_loss = validate_loss_only(model_core, val_loader, loss_fn)
    scheduler.step()

    if DEVICE == "cuda": torch.cuda.synchronize()
    print(f"[{epoch:03d}] train={tr_loss:.4f} val={val_loss:.4f} "
          f"lr={optimizer.param_groups[0]['lr']:.3e} time={(time.time()-t0):.1f}s")

    # record
    train_losses.append(float(tr_loss))
    val_losses.append(float(val_loss))

    # checkpoints
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model_core.state_dict(), SAVE_DIR / SAVE_FILE)

----- EPOCH 1 -----
[001] train=1.5450 val=1.5651 lr=1.200e-04 time=42.8s
----- EPOCH 2 -----
[002] train=1.4357 val=1.5622 lr=1.800e-04 time=2.8s
----- EPOCH 3 -----
[003] train=1.3274 val=1.4832 lr=2.400e-04 time=2.7s
----- EPOCH 4 -----
[004] train=1.2097 val=1.3683 lr=3.000e-04 time=2.7s
----- EPOCH 5 -----
[005] train=1.1300 val=1.2834 lr=3.000e-04 time=2.6s
----- EPOCH 6 -----
[006] train=1.0851 val=1.1352 lr=2.999e-04 time=3.0s
----- EPOCH 7 -----
[007] train=1.0538 val=1.0775 lr=2.997e-04 time=2.7s
----- EPOCH 8 -----
[008] train=1.0234 val=1.0442 lr=2.993e-04 time=2.7s
----- EPOCH 9 -----
[009] train=1.0011 val=1.0509 lr=2.987e-04 time=2.8s
----- EPOCH 10 -----
[010] train=0.9798 val=1.0176 lr=2.980e-04 time=2.6s
----- EPOCH 11 -----
[011] train=0.9549 val=0.9969 lr=2.971e-04 time=3.0s
----- EPOCH 12 -----
[012] train=0.9344 val=0.9928 lr=2.960e-04 time=2.9s
----- EPOCH 13 -----
[013] train=0.9137 val=0.9781 lr=2.948e-04 time=2.8s
----- EPOCH 14 -----
[014] train=0.8909 val=0.

# PLOT

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,3.5))
plt.plot(range(1, len(train_losses)+1), train_losses, label="train")
plt.plot(range(1, len(val_losses)+1),   val_losses,   label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.grid(True); plt.legend()
plt.title("Train vs Val Loss:" + TITLE)
plt.tight_layout()
plt.show()

# METRICS EVAL

In [ ]:
# load best
model_core.load_state_dict(torch.load(SAVE_DIR / SAVE_FILE, map_location=DEVICE))
model_for_eval = model_core.eval()

# run your evaluation utility
print(TITLE)
evaluate_and_print(model_for_eval, test_dataloader=test_loader, device=DEVICE, threshold=0.5, compute_auc=True)

# VISUALIZATION

In [ ]:
visualize_samples(
    model=model_for_eval,
    dataloader=test_loader,
    n_rows=10,
    device=DEVICE,
    threshold=0.5,
    clamp_pred_with_fov=True,
    figsize_per_row=(12, 3),
)